  # 00 - Oven Analysis

  Parses all furnace log files in `Raw oven*/`, plots temperature vs time, and exports a plateau summary table
  sampled every 5 min.  
  
  Verify the plateau before running Stage 01.

  **Input:** `Raw oven*/*.txt`
  **Output:** `Results/{condition}/Oven/`

## Quick links
- [Configuration](#configuration): `SAMPLE_ID`, `CONDITION_FILTER`, `TABLE_INTERVAL_S`
- [Import](#import): libraries, sample directory, conditions list
- [Oven plots generation](#oven-plots-generation): oven plots + plateau tables saved per condition
- [Summary](#summary): overview of all processed conditions

## Configuration 


**edit only the following cell**


In [ ]:

SAMPLE_ID = "SAMPLE_ID"   # sample folder name inside EIS-PIPELINE/

# Optional: process only specific gas conditions (leave empty [] to process ALL)
CONDITION_FILTER = []  # e.g. ["condition folder name (i.e. SampleID_Ar_SCCM_O2_SCCM_Tmax_Tmin_deltaT)"]

# Plateau_table sampling interval [seconds]
TABLE_INTERVAL_S = 300  # (default: every 5 minutes) 

## Import 

In [ ]:
import sys 
from pathlib import Path 

NOTEBOOK_DIR = Path().resolve() # find the notebook's directory 
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from pipeline.matching import find_furnace_log, parse_oven_file, plot_oven, extract_plateau_table
import pandas as pd

SAMPLE_DIR = NOTEBOOK_DIR / SAMPLE_ID

# Locate 'Raw oven' folder (handles trailing space in folder name)
raw_oven_candidates = list(SAMPLE_DIR.glob("Raw oven*"))
if not raw_oven_candidates:
    raise FileNotFoundError(f"No 'Raw oven' folder found in {SAMPLE_DIR}")
raw_oven_dir = raw_oven_candidates[0]

# Locate 'Raw data' folder to know which conditions exist
raw_data_dir = SAMPLE_DIR / "Raw data"
all_conditions = sorted([d.name for d in raw_data_dir.iterdir() if d.is_dir()])

if CONDITION_FILTER:
    conditions = [c for c in all_conditions if c in CONDITION_FILTER]
else:
    conditions = all_conditions

print(f"Sample     : {SAMPLE_ID}")
print(f"Raw oven   : {raw_oven_dir.name}")
print(f"Conditions to process ({len(conditions)}):")
for c in conditions:
    print(f"  {c}")

## Oven plots generation

In [ ]:
# Process all conditions
summary_rows = []

for condition_folder in conditions:
    print(f"\n{'='*70}")
    print(f"Condition: {condition_folder}")
    print(f"{'='*70}")

    results_dir = SAMPLE_DIR / "Results" / condition_folder / "Oven"
    results_dir.mkdir(parents=True, exist_ok=True)

    # Find furnace log
    try:
        furnace_log_path = find_furnace_log(SAMPLE_DIR, condition_folder)
        print(f"Furnace log: {furnace_log_path.name}")
    except FileNotFoundError as e:
        print(f"  [SKIP] {e}")
        continue

    # Parse
    parsed = parse_oven_file(furnace_log_path)
    df = parsed["df"]
    plateau_df_all = df[(df["Tsample"] >= 390) & (df["Tsample"] <= 610)]
    print(f"Start dt   : {parsed['start_dt']}")
    print(f"End dt     : {df['abs_datetime'].iloc[-1]}")
    print(f"T range    : {df['Tsample'].min():.1f} — {df['Tsample'].max():.1f} °C")
    if len(plateau_df_all) > 0:
        print(f"pO2 range  : {plateau_df_all['pO2'].min():.6f} — {plateau_df_all['pO2'].max():.6f} bar  (plateau only)")
    else:
        print(f"pO2 range  : n/a (no plateau data)")

    # Plot
    pdf_path = results_dir / f"oven_plot_{condition_folder}.pdf"
    plot_oven(parsed, save_path=pdf_path, show=True)

    # Plateau table
    plateau_df = extract_plateau_table(parsed, interval_s=TABLE_INTERVAL_S)
    csv_path = results_dir / f"plateau_table_{condition_folder}.csv"
    plateau_df.to_csv(csv_path, index=False)
    print(f"Plateau table: {len(plateau_df)} rows saved -> {csv_path.name}")

    # Show plateau window only (T in [390, 610])
    plateau_valid = plateau_df[
        (plateau_df["Tsample (°C)"] >= 390) & (plateau_df["Tsample (°C)"] <= 610)
    ]
    print(f"\nPlateau window ({len(plateau_valid)} rows):")
    pd.set_option("display.float_format", "{:.6f}".format)
    pd.set_option("display.max_rows", 100)
    display(plateau_valid.reset_index(drop=True))

    summary_rows.append({
        "condition":   condition_folder,
        "start_dt":    str(parsed['start_dt']),
        "end_dt":      str(df['abs_datetime'].iloc[-1]),
        "T_min":       round(df['Tsample'].min(), 1),
        "T_max":       round(df['Tsample'].max(), 1),
        "pO2_min":     plateau_df_all["pO2"].min() if len(plateau_df_all) > 0 else float("nan"),
        "pO2_max":     plateau_df_all["pO2"].max() if len(plateau_df_all) > 0 else float("nan"),
        "n_points":    len(df),
        "furnace_log": furnace_log_path.name,
    })

print(f"\n{'='*70}")
print(f"Stage 0 complete — {len(summary_rows)} condition(s) processed.")

## Summary

In [ ]:
# Summary table across all conditions
if summary_rows:
    df_summary = pd.DataFrame(summary_rows)
    print("Summary:")
    display(df_summary)

**Next step:** verify the plots and plateau tables, then run [01_ism_labeling.ipynb](01_ism_labeling.ipynb#01---ism-labeling)